# Pre-submission survey analysis

The eligible sample contains 4 respondents. Results use question-specific valid denominators; missingness remains separate from code absence. Multi-select and code counts can sum above the valid-response denominator because one respondent can contribute to several categories. Q2 (expected regulatory pathway), Q3 (clinical-panel text) and Q4 (programme role) are omitted from the public data. Q26 (constraint detail, adding little beyond Q25) and Q29 (streamlined-pathway discussion) remain in the processed data but are excluded from this analysis after results review. Fields omitted from processed/public inputs: Q33–Q34 (follow-up contact items).

Run all cells in order. This notebook reads processed aggregates and writes two result tables under `outputs/`. Plots display inline. For even samples, the numerical median averages the two central scores; it can lie between Likert categories. The full category distributions are retained.

All tables and figure labels show counts and valid-response denominators only. The `percent` field in the derived results CSV is empty for schema compatibility. Numerical medians, ranges and coding-coverage summaries are retained.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data/processed"
OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)

SURVEY = "presub"
# Results-review exclusions affect analysis only; the processed data retain these questions.
ANALYSIS_EXCLUDED = {26, 29}
counts = pd.read_csv(DATA / "survey_counts.csv", dtype={"category_id": str})
counts = counts.loc[counts.survey_id == SURVEY].copy()
assert ANALYSIS_EXCLUDED <= set(counts.question_id)
counts = counts.loc[~counts.question_id.isin(ANALYSIS_EXCLUDED)].copy()
coverage = pd.read_csv(DATA / "survey_code_coverage.csv")
coverage = coverage.loc[(coverage.survey_id == SURVEY) & ~coverage.question_id.isin(ANALYSIS_EXCLUDED)]
assert not counts.duplicated(["question_id", "category_id"]).any()
assert counts["sample_n"].eq(4).all()
assert counts.n.ge(0).all() and counts.n.le(counts.valid_n).all()
assert counts.applicable_n.eq(counts.sample_n-counts.skip_n-counts.not_applicable_n-counts.excluded_n).all()
assert counts.valid_n.eq(counts.applicable_n-counts.blank_n-counts.explicit_missing_n).all()
results, summaries = {}, {}


In [ ]:
%matplotlib inline
import textwrap
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Rectangle

# Scientific figure style v1.1, with count-scaled Likert stacks and small-sample bars.
try:
    font_manager.findfont("Arial", fallback_to_default=False)
    font = "Arial"
except ValueError:
    font = "DejaVu Sans"
    print("Arial unavailable; using DejaVu Sans.")
plt.rcdefaults()
plt.rcParams.update({"font.family": font, "font.size": 11, "figure.facecolor": "white",
    "axes.facecolor": "white", "axes.edgecolor": "#404040", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "grid.color": "#DDDDDD", "grid.linewidth": 0.6, "legend.fontsize": 10})
COLORS = ["#2E5661", "#89ACB8", "#D8D4C7", "#F2C48D", "#E6A052"]
DENOMINATORS = ["sample_n", "applicable_n", "valid_n", "blank_n", "explicit_missing_n", "skip_n", "not_applicable_n", "excluded_n"]

def show_table(table):
    display(table[DENOMINATORS].iloc[:1])
    display(table[["category_label", "n", "n/N"]])

def plot_question(table):
    valid = int(table.valid_n.iloc[0])
    kind = table.question_type.iloc[0]
    if valid < 2 or table.n.gt(0).sum() <= 1:
        print("Figure omitted: fewer than two valid responses or observed categories.")
        return
    t = table.loc[table.n > 0] if kind == "coded" else table
    labels = t.category_label.astype(str).tolist()
    heading = None
    if (SURVEY, int(t.question_id.iloc[0])) in [("postsub", 18), ("presub", 15)]:
        heading = "Number of changes"
        labels = [s.replace(" changes", "").replace(" change", "") for s in labels]
    fractions = [f"n = {n}/{valid}" for n in t.n]
    if kind == "numeric":
        values = np.repeat(t.numeric_value.to_numpy(), t.n.to_numpy())
        levels = pd.Series(values).groupby(values).cumcount()
        fig, ax = plt.subplots(figsize=(10, 3), layout="constrained")
        ax.scatter(values, levels, s=36, color="#5F828E", edgecolors="#404040", linewidths=0.6)
        ax.set(xlabel="Discussion duration (minutes)", xticks=np.unique(values), yticks=[], ylim=(-0.6, max(levels)+0.8))
        ax.spines["left"].set_visible(False)
    elif kind == "likert" and valid > 5:
        labels = [textwrap.fill(s, 38, break_long_words=False) for s in labels]
        fig, ax = plt.subplots(figsize=(10, 3 + 0.18*sum(s.count("\n")+1 for s in labels)), layout="constrained")
        left, bars, texts = 0, [], []
        for n, label, color, fraction in zip(t.n, labels, COLORS, fractions):
            bar = ax.barh(0, n, left=left, height=0.5, color=color, edgecolor="#404040", linewidth=0.6, label=label)[0]
            bars.append(bar)
            if n:
                texts.append((ax.text(left+n/2, 0, fraction, ha="center", va="center", fontsize=10,
                    color="white" if color == COLORS[0] else "#202020"), bar))
            left += n
        ax.add_patch(Rectangle((0, -0.25), valid, 0.5, fill=False, edgecolor="#404040", linewidth=0.6, clip_on=False))
        ax.set(xlim=(0, valid), xticks=range(valid+1), xlabel="Valid respondents (n)", yticks=[], ylim=(-0.6, 0.6))
        ax.spines["left"].set_visible(False)
        ax.xaxis.grid(True)
        ax.legend(handles=bars, labels=labels, loc="upper center", bbox_to_anchor=(0.5, -0.35), ncols=2,
            fancybox=False, framealpha=1, edgecolor="#404040")
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        for text, bar in texts:
            text.set_visible(text.get_window_extent(renderer).width+6 <= bar.get_window_extent(renderer).width)
    else:
        vertical = len(t) <= 5
        labels = [textwrap.fill(s, 20 if vertical else 44, break_long_words=False, break_on_hyphens=False) for s in labels]
        height = 4+0.18*max(s.count("\n") for s in labels) if vertical else max(3, 1+0.31*sum(s.count("\n")+1 for s in labels))
        fig, ax = plt.subplots(figsize=(10, height), layout="constrained")
        maximum = int(t.n.max())
        if vertical:
            bars = ax.bar(range(len(t)), t.n, width=0.65, color="#5F828E", edgecolor="#404040", linewidth=0.6)
            ax.set(xticks=range(len(t)), xticklabels=labels, ylabel="Respondents (n)", ylim=(0, maximum+0.8), yticks=range(maximum+1))
            if heading: ax.set_xlabel(heading)
            ax.yaxis.grid(True)
        else:
            bars = ax.barh(range(len(t)), t.n, height=0.65, color="#5F828E", edgecolor="#404040", linewidth=0.6)
            ax.set(yticks=range(len(t)), yticklabels=labels, xlabel="Respondents (n)", xlim=(0, maximum+0.8), xticks=range(maximum+1))
            if heading: ax.set_ylabel(heading)
            ax.invert_yaxis()
            ax.xaxis.grid(True)
            fractions = [s.replace("\n", " ") for s in fractions]
        ax.bar_label(bars, labels=fractions, padding=4, fontsize=10)
    display(fig)
    plt.close(fig)


## Q1. Did your organisation hold a Pre-Sub/Q-Sub meeting that focused primarily on a PCCP?

In [ ]:
table = counts.loc[counts.question_id == 1].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[1] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[1] = summary
plot_question(table)


## Q5. What was your primary reason for requesting FDA feedback on a PCCP at the Pre-Sub stage?

In [ ]:
table = counts.loc[counts.question_id == 5].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[5] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[5] = summary
plot_question(table)


## Q6. Which secondary motivations applied? (Choose as many as you like)

In [ ]:
table = counts.loc[counts.question_id == 6].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[6] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summary["total_selections"] = int(table.n.sum())
print("Total selections:", summary["total_selections"])
summaries[6] = summary
plot_question(table)


## Q7. Which interaction formats did you use with FDA? (Choose as many as you like)

*Note: One respondent selected both “Written only” and “Teleconference”; selections are reported as recorded.*


In [ ]:
table = counts.loc[counts.question_id == 7].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[7] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summary["total_selections"] = int(table.n.sum())
print("Total selections:", summary["total_selections"])
summaries[7] = summary
plot_question(table)


## Q8. Approximately how long (in minutes) was the live discussion, if any?

In [ ]:
table = counts.loc[counts.question_id == 8].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[8] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
values = np.repeat(table.numeric_value.to_numpy(), table.n.to_numpy())
assert len(values) == summary["valid_n"]
summary.update(median=float(np.median(values)), minimum=float(values.min()), maximum=float(values.max()))
display(pd.DataFrame([{k: summary[k] for k in ["median", "minimum", "maximum"]}]))
summaries[8] = summary
plot_question(table)


## Q9. How clear and useful was FDA’s combined written and verbal feedback?

Higher scores indicate greater clarity and usefulness.

In [ ]:
table = counts.loc[counts.question_id == 9].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[9] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
values = np.repeat(table.numeric_value.to_numpy(), table.n.to_numpy())
assert len(values) == summary["valid_n"]
summary.update(median=float(np.median(values)), minimum=float(values.min()), maximum=float(values.max()))
display(pd.DataFrame([{k: summary[k] for k in ["median", "minimum", "maximum"]}]))
summaries[9] = summary
plot_question(table)


## Q10. Where was clarity most lacking?

In [ ]:
table = counts.loc[counts.question_id == 10].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[10] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[10] = summary
plot_question(table)


## Q11. How clearly did FDA explain which minor PCCP updates would not require a new submission?

Higher scores indicate greater clarity.

Scale: 1 = Very unclear; 2 = Somewhat unclear; 3 = Mostly unclear; 4 = Mostly clear; 5 = Very clear.


In [ ]:
table = counts.loc[counts.question_id == 11].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[11] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
values = np.repeat(table.numeric_value.to_numpy(), table.n.to_numpy())
assert len(values) == summary["valid_n"]
summary.update(median=float(np.median(values)), minimum=float(values.min()), maximum=float(values.max()))
display(pd.DataFrame([{k: summary[k] for k in ["median", "minimum", "maximum"]}]))
summaries[11] = summary
plot_question(table)


## Q12. Please give one example of a “minor” change FDA said would not need a new submission.

In [ ]:
table = counts.loc[counts.question_id == 12].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[12] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[12] = summary
plot_question(table)


## Q13. What insight did the Pre-Sub predominantly provide?

In [ ]:
table = counts.loc[counts.question_id == 13].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[13] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[13] = summary
plot_question(table)


## Q14. Please highlight one especially valuable or surprising insight you received.

In [ ]:
table = counts.loc[counts.question_id == 14].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[14] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[14] = summary
plot_question(table)


## Q15. How many distinct change categories do you expect to cover in your PCCP?

In [ ]:
table = counts.loc[counts.question_id == 15].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[15] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[15] = summary
plot_question(table)


## Q16. Please provide example categories you plan to include (e.g., model retraining, parameter tuning, dataset expansion, new sensor, UI update).

In [ ]:
table = counts.loc[counts.question_id == 16].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[16] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[16] = summary
plot_question(table)


## Q17. Which PCCP section is most challenging so far? (Choose as many as you like)

In [ ]:
table = counts.loc[counts.question_id == 17].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[17] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summary["total_selections"] = int(table.n.sum())
print("Total selections:", summary["total_selections"])
summaries[17] = summary
plot_question(table)


## Q18. Briefly explain why this section(s) is challenging.

In [ ]:
table = counts.loc[counts.question_id == 18].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[18] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[18] = summary
plot_question(table)


## Q19. Did FDA ask for extra testing or documents you felt were disproportionate to device risk?

In [ ]:
table = counts.loc[counts.question_id == 19].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[19] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[19] = summary
plot_question(table)


## Q20. Explanation accompanying Q19

Question: If “Yes,” please describe the additional evidence requested.

The respondent reporting a minor disproportionate request in Q19 described a request for further detail about the verification and validation processes. This was the sole applicable response (1/1); the other three respondents answered No in Q19. The complete result table is retained below.


In [ ]:
table = counts.loc[counts.question_id == 20].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[20] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[20] = summary
plot_question(table)


## Q21. Does your draft PCCP require lighter evidence for low-risk changes and deeper evidence for high-risk changes (risk-tiered)?

In [ ]:
table = counts.loc[counts.question_id == 21].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[21] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[21] = summary
plot_question(table)


## Q22. If you changed PCCP scope after the Pre-Sub, what drove that decision?

In [ ]:
table = counts.loc[counts.question_id == 22].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[22] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[22] = summary
plot_question(table)


## Q23. Compared with a traditional submission without a PCCP, how much effort will finalising the PCCP take?

Higher scores indicate greater expected effort.

In [ ]:
table = counts.loc[counts.question_id == 23].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[23] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
values = np.repeat(table.numeric_value.to_numpy(), table.n.to_numpy())
assert len(values) == summary["valid_n"]
summary.update(median=float(np.median(values)), minimum=float(values.min()), maximum=float(values.max()))
display(pd.DataFrame([{k: summary[k] for k in ["median", "minimum", "maximum"]}]))
summaries[23] = summary
plot_question(table)


## Q24. What is the main driver of this effort level?

In [ ]:
table = counts.loc[counts.question_id == 24].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[24] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[24] = summary
plot_question(table)


## Q25. Which single resource constraint is currently greatest?

In [ ]:
table = counts.loc[counts.question_id == 25].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[25] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[25] = summary
plot_question(table)


## Q27. If FDA required all devices in your regulatory pathway and clinical area to include a PCCP, what impact would that have on your project?

In [ ]:
table = counts.loc[counts.question_id == 27].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[27] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[27] = summary
plot_question(table)


## Q28. Briefly explain why you chose this impact level.

In [ ]:
table = counts.loc[counts.question_id == 28].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[28] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[28] = summary
plot_question(table)


## Q30. How much long-term benefit do you expect the PCCP to provide once the device is on the market?

Higher scores indicate greater expected long-term benefit.

In [ ]:
table = counts.loc[counts.question_id == 30].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[30] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
values = np.repeat(table.numeric_value.to_numpy(), table.n.to_numpy())
assert len(values) == summary["valid_n"]
summary.update(median=float(np.median(values)), minimum=float(values.min()), maximum=float(values.max()))
display(pd.DataFrame([{k: summary[k] for k in ["median", "minimum", "maximum"]}]))
summaries[30] = summary
plot_question(table)


## Q31. How likely are you to include a PCCP in the eventual marketing submission for this device?

In [ ]:
table = counts.loc[counts.question_id == 31].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[31] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[31] = summary
plot_question(table)


## Q32. If guidance stays as drafted, how challenging would it be to include a PCCP?

Higher scores indicate greater anticipated challenge.

In [ ]:
table = counts.loc[counts.question_id == 32].sort_values("category_order").copy()
table["percent"] = np.nan  # Retained empty for result-schema compatibility.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[32] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
values = np.repeat(table.numeric_value.to_numpy(), table.n.to_numpy())
assert len(values) == summary["valid_n"]
summary.update(median=float(np.median(values)), minimum=float(values.min()), maximum=float(values.max()))
display(pd.DataFrame([{k: summary[k] for k in ["median", "minimum", "maximum"]}]))
summaries[32] = summary
plot_question(table)


## Coding coverage and result tables

Coverage includes observed responses receiving zero final codes. It excludes missing responses.

In [ ]:
for qid, distribution in coverage.groupby("question_id"):
    values = np.repeat(distribution.codes_per_response.to_numpy(), distribution.n_responses.to_numpy())
    assert len(values) == summaries[qid]["valid_n"]
    assert values.sum() == results[qid].n.sum()
    summaries[qid].update(no_code_n=int((values == 0).sum()), mean_codes=float(values.mean()),
                         min_codes=int(values.min()), max_codes=int(values.max()))
summary_table = pd.DataFrame(summaries.values()).sort_values("question_id")
display(summary_table.loc[summary_table.question_type == "coded", ["question_id", "valid_n", "no_code_n", "mean_codes", "min_codes", "max_codes"]])
pd.concat(results.values()).to_csv(OUTPUT / f"{SURVEY}_results.csv", index=False)
summary_table.to_csv(OUTPUT / f"{SURVEY}_summary.csv", index=False)
